Organização das pastas auxiliares na camada Bronze

In [0]:
base_aux = "/Volumes/viagens_teste/bronze/arquivos/auxiliares"

pastas_aux = [
    f"{base_aux}/diarias/vianorte",
    f"{base_aux}/diarias/atlas",

    f"{base_aux}/cancelados/vianorte",
    f"{base_aux}/cancelados/atlas",

    f"{base_aux}/nao_voados/vianorte",
    f"{base_aux}/nao_voados/atlas"
]

for pasta in pastas_aux:
    dbutils.fs.mkdirs(pasta)

Leitura dos arquivos de diárias da camada Bronze

In [0]:
df_diarias_a = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/auxiliares/diarias/vianorte/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_diarias_b = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/auxiliares/diarias/atlas/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

Validação da estrutura dos dados

In [0]:
df_diarias_a.printSchema()
df_diarias_b.printSchema()

print(df_diarias_a.columns)
print(df_diarias_b.columns)

Padronização das colunas das bases de diárias

In [0]:
from pyspark.sql.functions import col, lit

df_diarias_a_padrao = df_diarias_a.select(
    col("Protocolo").alias("id_diaria"),
    col("Viajante").alias("viajante"),
    col("Inicio").alias("data_inicio"),
    col("Fim").alias("data_fim"),
    col("Centro Custo").alias("centro_custo_origem"),
    col("Qtd Dias").alias("qtd_dias_informado"),
    col("Valor Total").alias("valor")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
from pyspark.sql.functions import col, lit

df_diarias_b_padrao = df_diarias_b.select(
    col("ID Pagamento").alias("id_diaria"),
    col("Beneficiario").alias("viajante"),
    col("Data Inicial").alias("data_inicio"),
    col("Data Final").alias("data_fim"),
    col("Unidade").alias("centro_custo_origem"),
    col("Numero Diarias").alias("qtd_dias_informado"),
    col("Total Pago").alias("valor")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)

In [0]:
df_diarias_a_padrao.columns == df_diarias_b_padrao.columns

Limpeza e consolidação das bases de diárias

In [0]:
df_diarias_a_padrao = df_diarias_a_padrao.filter(
    col("id_diaria").isNotNull()
)

df_diarias_b_padrao = df_diarias_b_padrao.filter(
    col("id_diaria").isNotNull()
)

In [0]:
print("ViaNorte:", df_diarias_a_padrao.count())
print("Atlas Corporate:", df_diarias_b_padrao.count())

In [0]:
df_diarias = df_diarias_a_padrao.unionByName(
    df_diarias_b_padrao
)

In [0]:
print("Total de diárias:", df_diarias.count())

display(df_diarias)

Tratamento do valor monetário

In [0]:
from pyspark.sql.functions import regexp_replace, datediff, col

df_diarias = df_diarias.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$ ",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_diarias.select(
    "valor",
    "valor_numerico"
).show(20, truncate=False)

Validação da quantidade de diárias

In [0]:
df_diarias = df_diarias.withColumn(
    "qtd_dias_calculado",
    datediff(
        col("data_fim"),
        col("data_inicio")
    )
)

In [0]:
df_diarias.select(
    "id_diaria",
    "data_inicio",
    "data_fim",
    "qtd_dias_informado",
    "qtd_dias_calculado"
).show(30, truncate=False)

Verificação de divergências

In [0]:
df_diarias.filter(
    col("qtd_dias_informado") != col("qtd_dias_calculado")
).select(
    "id_diaria",
    "agencia",
    "qtd_dias_informado",
    "qtd_dias_calculado",
    "data_inicio",
    "data_fim"
).show(truncate=False)

Cálculo da quantidade de diárias

In [0]:
df_diarias = df_diarias.withColumn(
    "qtd_dias_calculado",
    datediff(
        col("data_fim"),
        col("data_inicio")
    ) + 1
)

In [0]:
df_diarias.filter(
    col("qtd_dias_informado") != col("qtd_dias_calculado")
).select(
    "id_diaria",
    "agencia",
    "qtd_dias_informado",
    "qtd_dias_calculado"
).show(truncate=False)

Criação de colunas analíticas

In [0]:
from pyspark.sql.functions import lit, month, when

df_diarias = df_diarias.withColumn(
    "tipo_despesa",
    lit("Diaria")
)

In [0]:
df_diarias = df_diarias.withColumn(
    "trimestre",
    when(
        month(col("data_inicio")).between(1, 3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_inicio")).between(4, 6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_inicio")).between(7, 9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_diarias.groupBy(
    "trimestre"
).count().show()

Padronização do centro de custo

In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/referencias/centro_custo/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [0]:
df_diarias = (
    df_diarias.alias("d")
    .join(
        df_map.alias("m"),
        (
            (col("d.agencia") == col("m.Agencia")) &
            (col("d.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "d.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

Validação da base consolidada

In [0]:
print("Total de diárias:", df_diarias.count())

print(
    "Centros de custo não mapeados:",
    df_diarias.filter(
        col("centro_custo").isNull()
    ).count()
)

In [0]:
df_diarias.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.silver.diarias")

Leitura dos arquivos de viagens canceladas

In [0]:
df_cancelados_a = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/auxiliares/cancelados/vianorte/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_cancelados_b = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/auxiliares/cancelados/atlas/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

Validação da estrutura dos dados

In [0]:
display(df_cancelados_a)
display(df_cancelados_b)

df_cancelados_a.printSchema()
df_cancelados_b.printSchema()

print(df_cancelados_a.columns)
print(df_cancelados_b.columns)

Padronização das bases de cancelamentos

In [0]:
from pyspark.sql.functions import col, lit

df_cancelados_a_padrao = df_cancelados_a.select(
    col("Solicitacao").alias("id_cancelamento"),
    col("Viajante").alias("viajante"),
    col("Data Solicitacao").alias("data_solicitacao"),
    col("Centro Custo").alias("centro_custo_origem"),
    col("Tipo Original").alias("tipo_original"),
    col("Valor Estimado").alias("valor"),
    col("Motivo").alias("motivo")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
df_cancelados_b_padrao = df_cancelados_b.select(
    col("Reserva").alias("id_cancelamento"),
    col("Passageiro").alias("viajante"),
    col("Criacao").alias("data_solicitacao"),
    col("CC").alias("centro_custo_origem"),
    col("Servico").alias("tipo_original"),
    col("Valor Previsto").alias("valor"),
    col("Justificativa").alias("motivo")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)

Validação da padronização

In [0]:
df_cancelados_a_padrao.columns == df_cancelados_b_padrao.columns

Limpeza e consolidação dos cancelamentos

In [0]:
df_cancelados_a_padrao = df_cancelados_a_padrao.filter(
    col("id_cancelamento").isNotNull()
)

df_cancelados_b_padrao = df_cancelados_b_padrao.filter(
    col("id_cancelamento").isNotNull()
)

df_cancelados = df_cancelados_a_padrao.unionByName(
    df_cancelados_b_padrao
)

In [0]:
print("ViaNorte:", df_cancelados_a_padrao.count())
print("Atlas Corporate:", df_cancelados_b_padrao.count())
print("Total:", df_cancelados.count())

display(df_cancelados)

Tratamento do valor monetário

In [0]:
from pyspark.sql.functions import regexp_replace

df_cancelados = df_cancelados.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$ ",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_cancelados.select(
    "valor",
    "valor_numerico"
).show(20, truncate=False)

Criação de colunas analíticas

In [0]:
from pyspark.sql.functions import lit, month, when

df_cancelados = df_cancelados.withColumn(
    "situacao",
    lit("Cancelado")
)

In [0]:
df_cancelados = df_cancelados.withColumn(
    "trimestre",
    when(
        month(col("data_solicitacao")).between(1, 3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_solicitacao")).between(4, 6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_solicitacao")).between(7, 9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_cancelados.groupBy(
    "trimestre"
).count().show()

Padronização do centro de custo

In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/referencias/centro_custo/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [0]:
df_cancelados = (
    df_cancelados.alias("c")
    .join(
        df_map.alias("m"),
        (
            (col("c.agencia") == col("m.Agencia")) &
            (col("c.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "c.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

Validação da base consolidada

In [0]:
print("Total de cancelamentos:", df_cancelados.count())

print(
    "Centros de custo não mapeados:",
    df_cancelados.filter(
        col("centro_custo").isNull()
    ).count()
)

Persistência da camada Silver

In [0]:
df_cancelados.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.silver.cancelados")

In [0]:
df_cancelados_teste = spark.table(
    "viagens_teste.silver.cancelados"
)

display(df_cancelados_teste)

Leitura dos arquivos de passagens não voadas

In [0]:
df_nao_voados_a = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/auxiliares/nao_voados/vianorte/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

df_nao_voados_b = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/auxiliares/nao_voados/atlas/*.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

Validação da estrutura dos dados

In [0]:
display(df_nao_voados_a)
display(df_nao_voados_b)

df_nao_voados_a.printSchema()
df_nao_voados_b.printSchema()

print(df_nao_voados_a.columns)
print(df_nao_voados_b.columns)

Padronização das bases de passagens não voadas

In [0]:
from pyspark.sql.functions import col, lit

df_nao_voados_a_padrao = df_nao_voados_a.select(
    col("Bilhete").alias("id_bilhete"),
    col("Viajante").alias("viajante"),
    col("Emissao").alias("data_emissao"),
    col("Data Voo").alias("data_voo"),
    col("Centro Custo").alias("centro_custo_origem"),
    col("Valor Bilhete").alias("valor"),
    col("Status").alias("situacao")
).withColumn(
    "agencia",
    lit("ViaNorte")
)

In [0]:
df_nao_voados_b_padrao = df_nao_voados_b.select(
    col("Ticket").alias("id_bilhete"),
    col("Nome").alias("viajante"),
    col("Data Emissao").alias("data_emissao"),
    col("Embarque Previsto").alias("data_voo"),
    col("Unidade").alias("centro_custo_origem"),
    col("Valor").alias("valor"),
    col("Situacao").alias("situacao")
).withColumn(
    "agencia",
    lit("Atlas Corporate")
)

Validação da padronização

In [0]:
df_nao_voados_a_padrao.columns == df_nao_voados_b_padrao.columns

Limpeza e consolidação das bases

In [0]:
df_nao_voados_a_padrao = df_nao_voados_a_padrao.filter(
    col("id_bilhete").isNotNull()
)

df_nao_voados_b_padrao = df_nao_voados_b_padrao.filter(
    col("id_bilhete").isNotNull()
)

df_nao_voados = df_nao_voados_a_padrao.unionByName(
    df_nao_voados_b_padrao
)

In [0]:
print("ViaNorte:", df_nao_voados_a_padrao.count())
print("Atlas Corporate:", df_nao_voados_b_padrao.count())
print("Total:", df_nao_voados.count())

Tratamento do valor monetário

In [0]:
from pyspark.sql.functions import regexp_replace

df_nao_voados = df_nao_voados.withColumn(
    "valor_numerico",
    regexp_replace(
        regexp_replace(
            regexp_replace(
                col("valor"),
                "R\\$ ",
                ""
            ),
            "\\.",
            ""
        ),
        ",",
        "."
    ).cast("decimal(18,2)")
)

In [0]:
df_nao_voados.select(
    "valor",
    "valor_numerico"
).show(20, truncate=False)

Criação de colunas analíticas

In [0]:
from pyspark.sql.functions import lit, month, when

df_nao_voados = df_nao_voados.withColumn(
    "tipo_despesa",
    lit("Aereo")
)

In [0]:
df_nao_voados = df_nao_voados.withColumn(
    "trimestre",
    when(
        month(col("data_voo")).between(1, 3),
        "Primeiro Trimestre"
    )
    .when(
        month(col("data_voo")).between(4, 6),
        "Segundo Trimestre"
    )
    .when(
        month(col("data_voo")).between(7, 9),
        "Terceiro Trimestre"
    )
    .otherwise(
        "Quarto Trimestre"
    )
)

In [0]:
df_nao_voados.groupBy("trimestre").count().show()

Padronização do centro de custo

In [0]:
df_map = spark.read.csv(
    "/Volumes/viagens_teste/bronze/arquivos/referencias/centro_custo/Centro_Custo_Map.csv",
    header=True,
    inferSchema=True,
    sep=";"
)

In [0]:
df_nao_voados = (
    df_nao_voados.alias("n")
    .join(
        df_map.alias("m"),
        (
            (col("n.agencia") == col("m.Agencia")) &
            (col("n.centro_custo_origem") == col("m.Valor_Origem"))
        ),
        "left"
    )
    .select(
        "n.*",
        col("m.Centro_Custo_Padrao").alias("centro_custo"),
        col("m.Diretoria").alias("diretoria")
    )
)

Validação da base consolidada

In [0]:
print("Total de não voados:", df_nao_voados.count())

print(
    "Centros de custo não mapeados:",
    df_nao_voados.filter(
        col("centro_custo").isNull()
    ).count()
)

Persistência da camada Silver

In [0]:
df_nao_voados.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("viagens_teste.silver.nao_voados")

In [0]:
df_nao_voados_teste = spark.table(
    "viagens_teste.silver.nao_voados"
)

display(df_nao_voados_teste)